In [0]:
df = spark.read.csv("workspace.us_flights.us_flights_2023", header=True, inferSchema=True)

In [0]:
# Carregando Tabela com DataFrame
df = spark.read.table("workspace.us_flights.us_flights_2023")
display(df)

In [0]:
# Conferindo o Schema
df.printSchema()

In [0]:
# CRIANDO DELAY TOTAL
# Tratando Dados Nulos, referenciar colunas e criar valores literais (fixos)
from pyspark.sql.functions import coalesce, col, lit
df = df.withColumn(
    "Total_Delay",
    coalesce(col("Dep_Delay"), lit(0)) +
    coalesce(col("Arr_Delay"), lit(0))
)

In [0]:
# Criando Status do Voo
from pyspark.sql.functions import when
df = df.withColumn(
    "Flight_Status",
    when(col("Total_Delay") <= 0, "On Time")
    .when(col("Total_Delay") <= 30, "Minor Delay")
    .otherwise("Major Delay")
)


In [0]:
# Criando Período do dia
df = df.withColumn(
    "Day_Period",
    when(col("DepTime_label") == "Morning", "Morning")
    .when(col("DepTime_label") == "Afternoon", "Afternoon")
    .when(col("DepTime_label") == "Evening", "Evenving")
    .otherwise("Night")
)

In [0]:
# Criando Tabela FATO (fact_flights)
# Armazena eventos mensuráveis do negócio

fact_flights = df.select(
    "FlightDate",
    "Airline",
    "Tail_Number",
    "Dep_Airport",
    "Arr_Airport",
    "Total_Delay",
    "Flight_Status",
    "Day_Period"
)

fact_flights.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("us_flights.fact_flights")

In [0]:
df = spark.read.table("workspace.us_flights.fact_flights")
display(df)

In [0]:
# Criando Dimensões 
# Dimensão Data
from pyspark.sql.functions import year, month, dayofmonth, dayofweek

dim_date = df.select("FlightDate").distinct() \
    .withColumn("Year", year("FlightDate")) \
    .withColumn("Month", month("FlightDate")) \
    .withColumn("Day", dayofmonth("FlightDate")) \
    .withColumn("DayOfWeek", dayofweek("FlightDate"))

dim_date.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("us_flights.dim_date")

In [0]:
df = spark.read.table("workspace.us_flights.dim_date")
display(df)

In [0]:
# Dimensão Airline
fact_df = spark.read.table("workspace.us_flights.fact_flights")
dim_airline = fact_df.select("Airline").distinct()

dim_airline.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("us_flights.dim_airline")

In [0]:
# Dimensão Airport
from pyspark.sql.functions import col
fact_df = spark.read.table("workspace.us_flights.fact_flights")
dim_airport = fact_df.select(
    col("Dep_Airport").alias("Airport_Code")
).distinct()

dim_airport.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("us_flights.dim_airport")

In [0]:
%sql
-- Criando View para Dashboard (Power BI)
CREATE OR REPLACE VIEW us_flights.vw_airline_dashboard AS 
SELECT
    f.FlightDate,
    d.Year,
    d.Month,
    f.Airline,
    f.Dep_Airport,
    f.Arr_Airport,
    f.Total_Delay,
    f.Flight_Status,
    f.Day_Period
FROM us_flights.fact_flights f
JOIN us_flights.dim_date d
ON f.FlightDate = d.FlightDate;



In [0]:
%sql
-- Análise SQL
-- Atraso Médio por Aeroporto
SELECT Dep_Airport,
  AVG(Total_Delay) AS Avg_Delay
FROM us_flights.fact_flights
GROUP BY Dep_Airport
ORDER BY Avg_Delay DESC;

In [0]:
%sql
-- Ranking de Companhias Mais Pontuais
SELECT Airline,
       AVG(Total_Delay) AS Avg_Delay
FROM us_flights.fact_flights
GROUP BY Airline
ORDER BY Avg_Delay ASC;

In [0]:
%sql
-- Tendência Mensal
SELECT year,
       Month,
       COUNT (*) AS Total_flight,
       AVG(Total_Delay) AS Avg_Delay
FROM us_flights.vw_airline_dashboard
GROUP BY Year, Month
ORDER BY Year, Month;

In [0]:
# Particionar por Ano
fact_flights.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("FlightDate") \
    .saveAsTable("us_flights.fact_flights_Date")

In [0]:
%sql
-- Otimizar Delta
OPTIMIZE us_flights.fact_flights
ZORDER BY (Airline, Dep_Airport);

In [0]:
%sql
-- Camada GOLD
-- Tabela Agregada Mensal
CREATE OR REPLACE TABLE us_flights.gold_airline_performance
USING DELTA
AS
SELECT
    YEAR(FlightDate) AS Year,
    MONTH(FlightDate) AS Month,
    Airline,
    Dep_Airport,
    Arr_Airport,
    COUNT(*) AS Total_Flights,
    AVG(Total_Delay) AS Avg_Delay,
    SUM(CASE WHEN Flight_Status = 'On Time' THEN 1 ELSE 0 END) 
        / COUNT(*) AS OnTime_Rate
FROM us_flights.fact_flights
GROUP BY 
  YEAR(FlightDate), 
  MONTH(FlightDate), 
  Airline, 
  Dep_Airport, 
  Arr_Airport;

In [0]:
%sql
-- View Executiva
CREATE OR REPLACE VIEW us_flights.vw_powerbi_dashboard AS
SELECT
    Year,
    Month,
    Airline,
    Dep_Airport,
    Arr_Airport,
    Total_Flights,
    ROUND(Avg_Delay, 2) AS Avg_Delay,
    ROUND(OnTime_Rate * 100, 2) AS OnTime_Percentage
FROM us_flights.gold_airline_performance;

In [0]:
%sql
CREATE OR REPLACE TABLE us_flights.gold_airline_performance AS
SELECT
    YEAR(FlightDate) AS Year,
    MONTH(FlightDate) AS Month,
    Airline,
    COUNT(*) AS Total_Flights,
    SUM(CASE WHEN Flight_Status = 'On Time' THEN 1 ELSE 0 END) AS OnTime_Count,
    AVG(Total_Delay) AS Avg_Delay
FROM us_flights.fact_flights
GROUP BY
    YEAR(FlightDate),
    MONTH(FlightDate),
    Airline;